# B-TRUST Phase 76 — steps-matched fine-tuning robustness experiment (PROPOSED; run only after the author's approval)
**Preregistered here, before running** (written 2026-11-19, after Phase 73 and the exploratory Phase 75 analyses were seen).
No LLM component and no API calls.

**Motivation.** In Phase 73, fine-tuning with a fixed 3 epochs collapsed on SANGO (413 items, about 21 optimisation steps per epoch in CV) and MOHOM
(126 items, about 7), giving 98–100% single-class predictions and G0 = FAIL, although both corpora PASS under TF-IDF. Phase 75 (exploratory)
found that no training-free or size-matched diagnostic predicted transformer transfer. Separating label recoverability from model trainability
requires retraining with a configuration adequate for small corpora.

**Single manipulated factor.** Epochs = max(3, ceil(300 / ceil(n_train / 16))), i.e. at least 300 optimisation steps per fit. Everything else is
identical to Phase 73: models, max_len 64, batch 16, lr 2e-5, weight decay 0.01, seed 70, linear schedule, no warmup, corpora, targets,
leakage rule, CV folds, G0 thresholds. All eight corpora are retrained, so results are comparable within this phase.

**Hypotheses (for each of DistilBERT and RoBERTa).**
- **R1 (collapse resolved; descriptive criterion):** for SANGO and MOHOM, the most frequent out-of-fold class holds < 80% of predictions.
- **R2 (gate decision; descriptive criterion):** SANGO and MOHOM receive G0 = PASS or CAUTION (not FAIL).
- **R3 (association; inferential):** Spearman ρ(G0 lift, mean transfer T) > 0 over the eight corpora; exact one-sided permutation test, α = 0.05.

**Pre-specified interpretation.** If R1 and R2 hold, the Phase 73 FAIL decisions for SANGO and MOHOM are attributed to training configuration, not labels.
If collapse persists despite ≥ 300 steps, the FAILs are attributed to the corpora at this model class (size and/or labels), without distinguishing which.
R3 tests whether the association holds when no corpus is collapsed. Phase 73 results are not replaced; both are reported.
**Record:** run date, GPU, library versions, this notebook's SHA-256.

**Amendment A1 (2026-11-20, before any run; author-approved requirements).** Added only artifact saving and diagnostics: mandatory Google Drive
output in a Phase 76-only folder (with a guard against Phase 73 paths), fp16 weights of the 16 final full-fit models, tokenisers and training
configurations, per-epoch training loss and post-training train accuracy for every fit, and an overfitting table. The training procedure,
data, splits, settings, hypotheses R1–R3 and interpretation rules are unchanged. Diagnostics are computed after training in eval mode and
do not alter any prediction. Cross-validation fold weights are not saved by default (about 15 GB); their predictions are saved.
**Correction:** the previous version (68ae8b96) still pointed its output folder to the Phase 73 location; it was never run.

## 1. Setup — GPU runtime; upload `BTRUST_Frozen_Supporting_Files_2026-11-07.zip` and `BTRUST_v2.8_*.zip` when prompted

In [ ]:
!pip -q install transformers==4.46.3 accelerate scikit-learn pandas openpyxl
import os, zipfile, glob, subprocess, sys, platform, torch, transformers
from google.colab import files
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
up = files.upload()
for z in up: zipfile.ZipFile(z).extractall('/content/in')
os.makedirs('/content/ex', exist_ok=True)
for z in glob.glob('/content/in/**/original_phase_archives/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall(f'/content/ex/{os.path.basename(z)[:-4]}')
os.environ['BTRUST_EX'] = '/content/ex'
code_dir = glob.glob('/content/in/**/v2.0_phases65-68/code', recursive=True)[0]
FARAJI = glob.glob('/content/in/**/data_faraji_MIT', recursive=True)[0]
os.chdir(code_dir); subprocess.run(['sh','fetch_and_prepare.sh'], check=True); sys.path.insert(0, code_dir)
# ---- Google Drive is REQUIRED (approved protocol): all outputs, configs and checkpoints are written there ----
from google.colab import drive; drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Google Drive is not mounted'
OUT = '/content/drive/MyDrive/BTRUST_p76_out'      # Phase 76 only; Phase 73 folders are never read or written
assert 'p73' not in OUT.lower(), 'Refusing to write Phase 76 outputs into a Phase 73 location'
os.makedirs(OUT, exist_ok=True)
with open(f'{OUT}/_write_test.txt', 'w') as fh: fh.write('ok')
SAVE_FULL_FIT_WEIGHTS = True   # 16 final models (8 corpora x 2), fp16 safetensors: about 1.1 GB DistilBERT + 2.0 GB RoBERTa
SAVE_CV_FOLD_WEIGHTS = False   # 80 cross-validation fold models would need about 15 GB; fold predictions, losses and train accuracy are always saved
import json as _json
_json.dump(dict(torch=torch.__version__, transformers=transformers.__version__, python=platform.python_version(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE',
                save_full_fit_weights=SAVE_FULL_FIT_WEIGHTS, save_cv_fold_weights=SAVE_CV_FOLD_WEIGHTS), open(f'{OUT}/environment.json', 'w'), indent=1)
print('Outputs ->', OUT)

## 2. Corpora — identical to Phase 71 (asserted)

In [ ]:
import numpy as np, pandas as pd, itertools, random, hashlib, json, time
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import corpora as C
REV = {"remember":1,"understand":2,"apply":3,"analyze":4,"evaluate":5,"create":6}
def faraji(f):
    d = pd.read_csv(f'{FARAJI}/{f}'); d = pd.DataFrame({"text": d.QUESTION, "y": d["BT LEVEL"].str.strip().str.lower().map(REV)})
    d = d[d.text.map(lambda s: isinstance(s, str) and s.strip() != "")].copy(); d["n"] = d.text.map(C.norm)
    return d.drop_duplicates("n").reset_index(drop=True)
corp = {"EDUB": C.edub(), "YAHYA": faraji("yahya-2012_revised.csv"), "SANGO": faraji("Sangodiah_2017_415_revised.csv"),
        "MOHOM": faraji("Mohammed-and-Omar_2020_126_revised.csv"), "GANI": faraji("Gani_2023_collected-dataset-1200_revised.csv"),
        "SCAR-I": C.scar_i(), "SCAR-E": C.scar_e(), "EDUQG": C.eduqg()}
yq, V = C.youleqd(); lk = C.lak(); TG = ["YAHYA","SANGO","MOHOM","GANI","SCAR-E","EDUQG"]
expected = {'EDUB':2550,'YAHYA':599,'SANGO':413,'MOHOM':126,'GANI':1185,'SCAR-I':2550,'SCAR-E':1839,'EDUQG':903}
assert {k: len(v) for k, v in corp.items()} == expected, {k: len(v) for k, v in corp.items()}
assert len(yq) == 70 and len(lk) == 40
h = lambda s: hashlib.sha256(s.encode()).hexdigest()[:16]   # item key = hash of normalised text (no text written to outputs)

## 3. Fine-tuning helper (Phase 73 helper; only the number of epochs changes, from the ≥ 300-step rule)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def seed_all(s=70): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
MIN_STEPS = 300
def n_epochs(n_train): return max(3, int(np.ceil(MIN_STEPS / int(np.ceil(n_train / 16)))))
EPOCH_LOG = []
DIAG = []   # per-fit diagnostics (does not affect training)
def fit_predict(name, Xtr, ytr, Xte_list, tag='', save_dir=None):
    seed_all(); tok = AutoTokenizer.from_pretrained(name)
    m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=6).to(dev)
    enc = tok(list(Xtr), truncation=True, max_length=64, padding=True, return_tensors='pt'); y = torch.tensor(np.asarray(ytr) - 1)
    idx = np.arange(len(y)); opt = torch.optim.AdamW(m.parameters(), lr=2e-5, weight_decay=0.01)
    E = n_epochs(len(y)); EPOCH_LOG.append((name, len(y), E))
    sch = get_linear_schedule_with_warmup(opt, 0, E * int(np.ceil(len(y) / 16))); m.train(); rg = np.random.default_rng(70)
    epoch_loss = []
    for ep in range(E):
        rg.shuffle(idx); tot, nb_ = 0.0, 0
        for i in range(0, len(idx), 16):
            b = idx[i:i+16]
            out = m(input_ids=enc['input_ids'][b].to(dev), attention_mask=enc['attention_mask'][b].to(dev), labels=y[b].to(dev))
            out.loss.backward(); opt.step(); sch.step(); opt.zero_grad()
            tot += float(out.loss.detach()); nb_ += 1
        epoch_loss.append(tot / max(nb_, 1))
    m.eval(); preds = []
    for X in Xte_list:
        P = []
        for i in range(0, len(X), 64):
            e = tok(list(X[i:i+64]), truncation=True, max_length=64, padding=True, return_tensors='pt').to(dev)
            with torch.no_grad(): P.append(m(**e).logits.argmax(-1).cpu().numpy() + 1)
        preds.append(np.concatenate(P) if P else np.array([], int))
    # ---- diagnostics after training (eval mode, no gradient; does not change any prediction above) ----
    Ptr = []
    for i in range(0, len(Xtr), 64):
        e = tok(list(Xtr[i:i+64]), truncation=True, max_length=64, padding=True, return_tensors='pt').to(dev)
        with torch.no_grad(): Ptr.append(m(**e).logits.argmax(-1).cpu().numpy() + 1)
    train_acc = float((np.concatenate(Ptr) == np.asarray(ytr)).mean())
    DIAG.append(dict(tag=tag, model=name, n_train=int(len(y)), epochs=E, steps=E * int(np.ceil(len(y) / 16)),
                     epoch_loss=[round(v, 5) for v in epoch_loss], train_acc=train_acc))
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        m.half().save_pretrained(save_dir, safe_serialization=True); tok.save_pretrained(save_dir)
        json.dump(dict(base_model=name, num_labels=6, label_order="index k -> revised level k+1", max_len=64, batch=16, lr=2e-5,
                       weight_decay=0.01, seed=70, scheduler="linear, no warmup", epochs=E, min_steps_rule=MIN_STEPS,
                       n_train=int(len(y)), saved_dtype="float16 (training was float32)", tag=tag), open(f"{save_dir}/b_trust_training_config.json", "w"), indent=1)
    del m; torch.cuda.empty_cache()
    return preds

## 4. Run — checkpointed after every (model, corpus); re-running this cell resumes where it stopped

In [ ]:
def lookup_cv(key, y):
    p = np.zeros(len(y), int); maj = pd.Series(y).mode()[0]
    for a, b in StratifiedKFold(5, shuffle=True, random_state=65).split(key, y):
        t = pd.DataFrame({'k': key[a], 'y': y[a]}).groupby('k').y.agg(lambda s: s.mode()[0]); p[b] = pd.Series(key[b]).map(t).fillna(maj).astype(int).to_numpy()
    return p
def run(name, short):
    for c, d in corp.items():
        done = f"{OUT}/gate_{short}_{c}.json"
        if os.path.exists(done): print("skip (done)", short, c); continue
        t0 = time.time(); X = d.text.to_numpy(); y = d.y.to_numpy(); p = np.zeros(len(y), int)
        for fk, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=65).split(X, y)):
            p[b] = fit_predict(name, X[a], y[a], [X[b]], tag=f'{short}|{c}|cv{fk}', save_dir=(f'{OUT}/models/cv/{short}_{c}_fold{fk}' if SAVE_CV_FOLD_WEIGHTS else None))[0]
        pd.DataFrame({"model": short, "corpus": c, "item_key": d.n.map(h), "y": y, "oof_pred": p}).to_csv(f"{OUT}/oof_{short}_{c}.csv", index=False)
        acc = (p == y).mean(); maj = pd.Series(y).value_counts(normalize=True).iloc[0]
        meta = (lookup_cv(d.meta.to_numpy(), y) == y).mean() if 'meta' in d else np.nan
        lift = acc - np.nanmax([maj, meta]); k = cohen_kappa_score(y, p)
        gate = "FAIL" if (lift < .05 or k < .20) else ("PASS" if (lift >= .15 and k >= .40) else "CAUTION")
        ts = set(d.n); tests, meta_t = [], []
        for tn in TG:
            if tn == c or {c, tn} == {"SCAR-I", "SCAR-E"}: continue
            t = corp[tn]; keep = ~t.n.isin(ts); tests.append(t.text[keep].to_numpy()); meta_t.append((tn, t[keep]))
        P = fit_predict(name, X, y, tests + [yq.text.to_numpy(), lk.text.to_numpy()], tag=f'{short}|{c}|full', save_dir=(f'{OUT}/models/full/{short}_{c}' if SAVE_FULL_FIT_WEIGHTS else None))
        rows = []; accs = {}
        for (tn, t), pp in zip(meta_t, P[:len(meta_t)]):
            accs[tn] = (pp == t.y.to_numpy()).mean()
            rows.append(pd.DataFrame({"target": tn, "item_key": t.n.map(h).values, "y": t.y.values, "pred": pp}))
        accs["YLQ70"] = (P[-2] == yq.y.to_numpy()).mean()
        rows.append(pd.DataFrame({"target": "YLQ70", "item_key": yq.n.map(h).values, "row": np.arange(70), "vid": yq.vid.values, "y": yq.y.values, "pred": P[-2]}))
        accs["LAK40"] = np.mean([a in s for a, s in zip(P[-1], lk.refs)])
        rows.append(pd.DataFrame({"target": "LAK40", "item_key": lk.n.map(h).values, "refs": [sorted(s) for s in lk.refs], "pred": P[-1]}))
        pd.concat(rows).assign(model=short, train=c).to_csv(f"{OUT}/items_{short}_{c}.csv", index=False)
        json.dump(dict(model=name, train=c, n=len(y), cv_acc=acc, G0_lift=lift, kappa_cv=k, G0=gate, **accs,
                       T=float(np.mean(list(accs.values()))), seconds=round(time.time() - t0), oof_top_class_share=float(pd.Series(p).value_counts(normalize=True).iloc[0]), epochs_full_fit=n_epochs(len(y))), open(done, "w"))
        pd.DataFrame([dd for dd in DIAG if dd['tag'].startswith(f'{short}|{c}|')]).to_json(f"{OUT}/diag_{short}_{c}.jsonl", orient='records', lines=True)
        print(short, c, gate, round(lift, 3), "T", round(np.mean(list(accs.values())), 3), f"{time.time()-t0:.0f}s")
for name, short in [("distilbert-base-uncased", "distilbert"), ("roberta-base", "roberta")]: run(name, short)

## 5. Tests (R1–R3) and output files

In [ ]:
summary = []
for short in ["distilbert", "roberta"]:
    R = pd.DataFrame([json.load(open(f"{OUT}/gate_{short}_{c}.json")) for c in corp]); R.to_csv(f"{OUT}/p76_k8_{short}.csv", index=False)
    x, yy = R.G0_lift.to_numpy(), R["T"].to_numpy(); obs = spearmanr(x, yy).statistic
    pv = np.mean([spearmanr(x, np.array(q)).statistic >= obs - 1e-12 for q in itertools.permutations(yy)])
    sep = bool(R[R.G0 == "PASS"]["T"].min() > R[R.G0 == "FAIL"]["T"].max()) if (R.G0 == "FAIL").any() and (R.G0 == "PASS").any() else None
    small = R.set_index("train").loc[["SANGO","MOHOM"]]
    summary.append(dict(model=short, rho=obs, exact_p=pv, R3_supported=bool(pv < 0.05),
                        R1_collapse_resolved=bool((small.oof_top_class_share < 0.80).all()), R2_not_FAIL=bool((small.G0 != "FAIL").all()),
                        SANGO_G0=small.loc["SANGO","G0"], MOHOM_G0=small.loc["MOHOM","G0"], PASS_above_FAIL=sep))
    display(R.round(4))
S = pd.DataFrame(summary); S.to_csv(f"{OUT}/p76_tests.csv", index=False); display(S)
# Overfitting diagnostics: train accuracy vs out-of-fold accuracy (CV folds) and full-fit train accuracy
rows = []
for short in ["distilbert", "roberta"]:
    for c in corp:
        dd = pd.read_json(f"{OUT}/diag_{short}_{c}.jsonl", lines=True); g = json.load(open(f"{OUT}/gate_{short}_{c}.json"))
        cv = dd[dd.tag.str.contains('|cv', regex=False)]
        rows.append(dict(model=short, corpus=c, epochs_cv=int(cv.epochs.iloc[0]), train_acc_cv_mean=cv.train_acc.mean(), oof_acc=g["cv_acc"],
                         gap=cv.train_acc.mean() - g["cv_acc"], final_epoch_loss_cv_mean=float(np.mean([l[-1] for l in cv.epoch_loss])),
                         train_acc_full=float(dd[dd.tag.str.endswith('|full')].train_acc.iloc[0]), T=g["T"]))
O = pd.DataFrame(rows); O.to_csv(f"{OUT}/p76_overfitting.csv", index=False); display(O.round(3))
# Results-only archive (no model weights) to send back; the weights stay on Google Drive
import shutil
res_dir = '/content/p76_results_only'; shutil.rmtree(res_dir, ignore_errors=True); shutil.copytree(OUT, res_dir, ignore=shutil.ignore_patterns('models', '*.zip'))
shutil.make_archive('/content/p76_outputs', 'zip', res_dir); shutil.copy('/content/p76_outputs.zip', f'{OUT}/p76_outputs.zip')
files.download('/content/p76_outputs.zip')